# v4 repaired + loubna's edits: line following, gaps, and green-marker turns

Based on the team's v4 green-marker design and the earlier gap controller. No motor movement starts automatically; uncomment the driving call deliberately. Run one cell at a time on the Jetson, never Run All. Shut down previous camera kernels first. Resolve power alarms before driving.

Copy working calibration from your robot: black threshold, bands, speeds, gains, reverse setting, and gap timeout. The defaults here are trial settings, not your measured values. Green-triggered turns reverse one wheel regardless of the ordinary-curve ALLOW_REVERSE setting.

Corrections: valid green detector with area/shape filtering and return value; OpenCV 3/4 contour compatibility; actual stop/advance/turn sequence; turn timers start with the turn; departure and multi-frame reacquisition; latched timeout stops; marker cooldown; restored gap handling and camera freshness checks.

Limits: marker geometry is heuristic. There is no full intersection topology detector or confirmation that a marker is before an intersection. U-turn angle is not measured: minimum duration and line reacquisition need calibration and can select the wrong branch. White gaps use bounded open-loop motion. Software timeouts do not protect against a frozen process/driver. Keep a physical stop accessible. Hi Loubna

## 1. Initialize camera once


In [ ]:
import time
import cv2 #카메라
import numpy as np # 
import traitlets #값 바뀔 때 신호 연동
import ipywidgets as widgets #화면 위젯

from IPython.display import display #
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()

camera = Camera.instance(width=224, height=224)

live_view = widgets.Image(format="jpeg", width=336, height=336) #camera가 보여주는 영상 만들 공간

camera_link = traitlets.dlink(
    (camera, "value"),
    (live_view, "value"),
    transform=bgr8_to_jpeg
)

display(live_view)